# 🎙️ Voice Studio — запуск в Google Colab (бесплатно, с GPU)

Озвучка ролика одним и тем же голосом + распознавание речи + липсинк.

## Как пользоваться — 3 действия:
1. **Включи GPU** (бесплатный, чтобы липсинк был быстрым):
   сверху меню **Среда выполнения → Сменить среду выполнения → Аппаратный ускоритель: GPU (T4) → Сохранить**.
2. Сверху меню **Среда выполнения → Выполнить все** (Run all).
3. Жди, пока внизу последней ячейки появится **публичная ссылка** вида `https://xxxx.gradio.live` — открывай её, это твоя рабочая Voice Studio. 🚀

> Первый запуск качает модели — это несколько минут. Ссылка живёт, пока открыт этот ноутбук.
> Если что-то ругнётся красным — скопируй текст ошибки и пришли мне, поправим.

### Шаг 1. Проверка GPU и загрузка кода

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu or '⚠️ GPU не подключён! Включи: Среда выполнения → Сменить среду выполнения → GPU')

# Всегда берём СВЕЖУЮ копию кода (надёжнее, чем pull — не путается с папками).
# Модели XTTS/Whisper кэшируются вне репо (~/.local), поэтому не перекачиваются.
%cd /content
!rm -rf /content/lama
!git clone --branch claude/russian-greeting-j4vxeb https://github.com/amalsabitov98-art/lama.git
%cd /content/lama/voice-studio
print('\n✅ Код на месте (свежая копия)')

### Шаг 2. Установка зависимостей (озвучка + распознавание речи)
Несколько минут — качается TTS-движок и Whisper.

In [ ]:
!apt-get -qq install -y ffmpeg
# coqui-tts — форк XTTS (импорт TTS); transformers пиним <5.0, иначе coqui-tts
# падает с ImportError (isin_mps_friendly). openai-whisper — распознавание речи.
!pip install -q gradio openai-whisper coqui-tts "transformers>=4.47,<5.0"
print('✅ Озвучка и распознавание готовы')

### Шаг 3. Липсинк (Wav2Lip) — подгонка губ под речь
На бесплатном GPU работает быстро. Скачиваем модель и веса.

> Если веса не скачаются (миррор отвалился) — не страшно: приложение всё равно
> запустится и озвучит ролик твоим голосом, просто без подгонки губ. Напиши мне —
> дам рабочую ссылку на веса.

In [ ]:
import os
os.makedirs('third_party', exist_ok=True)
!cd third_party && (git clone https://github.com/Rudrabha/Wav2Lip.git 2>/dev/null || echo 'уже есть')
!mkdir -p third_party/Wav2Lip/checkpoints third_party/Wav2Lip/face_detection/detection/sfd

# Веса Wav2Lip (мирроры сообщества)
!wget -q -O third_party/Wav2Lip/checkpoints/wav2lip_gan.pth 'https://huggingface.co/camenduru/Wav2Lip/resolve/main/wav2lip_gan.pth' || echo '⚠️ wav2lip_gan.pth не скачался'
!wget -q -O third_party/Wav2Lip/face_detection/detection/sfd/s3fd.pth 'https://huggingface.co/camenduru/Wav2Lip/resolve/main/s3fd.pth' || echo '⚠️ s3fd.pth не скачался'

# Зависимости Wav2Lip (librosa НЕ пиним — coqui-tts требует >=0.11, иначе конфликт)
!pip install -q opencv-python numba

# Проверка, что веса на месте
w = 'third_party/Wav2Lip/checkpoints/wav2lip_gan.pth'
sz = os.path.getsize(w) if os.path.exists(w) else 0
print(f'✅ Липсинк готов, вес {sz//1024//1024} МБ' if sz > 1000000 else '⚠️ Веса липсинка не докачались — приложение запустится без липсинка (напиши мне)')

### Шаг 4. Запуск Voice Studio 🚀
Когда появится ссылка **https://….gradio.live** — открывай её. Это твоё приложение.

In [ ]:
import os
os.environ['VOICE_STUDIO_SHARE'] = '1'  # публичная ссылка gradio.live
%cd /content/lama/voice-studio
!python app.py